In [10]:
import numpy as np
import pandas as pd

All .str methods work element-wise on entire column, not just one string

.str.lower() → Converts all strings to lowercase

 .str.upper() → Converts all strings to uppercase

 .str.capitalize() → Capitalizes first letter of each string

 .str.strip() → Removes leading and trailing spaces

 .str.split(" ") → Splits string into list based on delimiter (default = space)

 .str.contains(val, case=False) → Checks if string contains a value (case-insensitive if case=False)

DATA TRANSFORMATION

Step 1: Load Data

In [11]:
df = pd.read_csv("raw_data.csv")
df

,id,name,age,country,gender,income
0,1,John Doe,29.0,USA,Male,55000.0
1,1,John Doe,29.0,USA,Male,55000.0
2,2,Jane Smith,NaN,Canada,Female,62000.0
3,3,Alex,NaN,USA,Unknown,47000.0
4,4,Maria Garcia,34.0,Spain,Female,NaN
5,5,Li Wei,27.0,China,Male,51000.0
6,6,NaN,45.0,India,Female,73000.0
7,7,Ahmed Khan,38.0,NaN,Male,68000.0
8,8,Rachel Lee,29.0,USA,Female,62000.0
9,9,Carlos Ruiz,NaN,Mexico,Male,45000.0


Step 2: Check Missing Values

In [12]:
# Count missing values in each column
df.isnull().sum()

id         0
name       1
age        3
country    1
gender     1
income     1
dtype: int64

Step 3: Handle Missing Values

In [ ]:
df2 = df.copy()

# Fill missing income using forward fill
df2['income'] = df2['income'].ffill()

# ✅ Fill missing age with mean
df2['age'] = df2['age'].fillna(df2['age'].mean())

# ✅ Fill missing categorical values
df2['name'] = df2['name'].fillna("Unknown")
df2['country'] = df2['country'].fillna("Not Specified")
df2['gender'] = df2['gender'].fillna("Not Specified")



Step 4: Remove Duplicates

In [14]:
df2 = df2.drop_duplicates()

Step 5: Transform Data (Feature Engineering)

In [15]:
df2['tax'] = df2['income'].apply(lambda x: '20%' if x < 50000 else '25%')

In [ ]:
# ✅ Using map()
gender_map = {'Male': 0, 'Female': 1, 'Unknown': 2, 'Not Specified': 3}
df2['gender_code'] = df2['gender'].map(gender_map)

# ✅ Create new column using assign()  # assign() → Adds or modifies columns without changing original DataFrame
df2 = df2.assign(income_in_lakhs = df2['income'] / 100000)

# ✅ Replace values
df2['country'] = df2['country'].replace("USA", "United States")

🔃 Step 6: Sorting & Index

In [ ]:
df2 = df2.sort_values(by='income', ascending=False)  # sort_values() → Sorts DataFrame based on column values

df2 = df2.reset_index(drop=True)  # reset_index() → Resets index to default integer index

In [18]:
df2

,id,name,age,country,gender,income,tax,gender_code,income_in_lakhs
0,6,Unknown,45.00,India,Female,73000.0,25%,1,0.73
1,7,Ahmed Khan,38.00,Not Specified,Male,68000.0,25%,0,0.68
2,8,Rachel Lee,29.00,United States,Female,62000.0,25%,1,0.62
3,2,Jane Smith,32.75,Canada,Female,62000.0,25%,1,0.62
4,10,Emily Davis,31.00,United States,Not Specified,58000.0,25%,3,0.58
5,1,John Doe,29.00,United States,Male,55000.0,25%,0,0.55
6,5,Li Wei,27.00,China,Male,51000.0,25%,0,0.51
7,4,Maria Garcia,34.00,Spain,Female,47000.0,20%,1,0.47
8,3,Alex,32.75,United States,Unknown,47000.0,20%,2,0.47
9,9,Carlos Ruiz,32.75,Mexico,Male,45000.0,20%,0,0.45


GROUP BY AND AGGREGATION FUNCTIONs

In [ ]:
#Country wise grouping
df2.groupby('country')['income'].mean()
df2.groupby('country')['income'].min()
df2.groupby('country')['income'].max()


country
Canada           62000.0
China            51000.0
India            73000.0
Mexico           45000.0
Not Specified    68000.0
Spain            47000.0
United States    47000.0
Name: income, dtype: float64

In [21]:
# Gender Wise Grouping

df2.groupby('gender')['income'].max()

gender
Female           73000.0
Male             68000.0
Not Specified    58000.0
Unknown          47000.0
Name: income, dtype: float64

In [ ]:
# Calculating everything using aggregate function

df2.groupby('country')['income'].aggregate(['mean', 'median','max','min'])

,mean,median,max,min
country,,,,
Canada,62000.0,62000.0,62000.0,62000.0
China,51000.0,51000.0,51000.0,51000.0
India,73000.0,73000.0,73000.0,73000.0
Mexico,45000.0,45000.0,45000.0,45000.0
Not Specified,68000.0,68000.0,68000.0,68000.0
Spain,47000.0,47000.0,47000.0,47000.0
United States,55500.0,56500.0,62000.0,47000.0


In [24]:
df2.groupby('country').aggregate({
    'income': 'mean',
    'age': 'max'
})

df2.groupby('country').aggregate(
    mean_salary = ('income','mean'),
    maximum_age = ('age', 'max')
)

,mean_salary,maximum_age
country,,
Canada,62000.0,32.75
China,51000.0,27.00
India,73000.0,45.00
Mexico,45000.0,32.75
Not Specified,68000.0,38.00
Spain,47000.0,34.00
United States,55500.0,32.75


Melt = “Melting columns” → columns collapse into rows

Pivot = “Spreading data” → rows expand into columns

🔄 Melt (Wide → Long)
# melt(id_vars, value_vars, var_name, value_name)
# → Converts columns into rows (unpivot data)

🧠 Meaning:

id_vars → columns to keep

value_vars → columns to unpivot

var_name → new column (old column names)

value_name → new column (values)

In [ ]:
df = pd.DataFrame({
    'name': ['A', 'B'],
    'math': [90, 80],
    'science': [85, 88]
})

df_melted = df.melt(
    id_vars='name',
    value_vars=['math', 'science'],
    var_name='subject',
    value_name='marks'
)

df_melted

🔄 Pivot (Long → Wide)
# pivot(index, columns, values)
# → Converts rows into columns

🧠 Meaning:

index → rows to keep

columns → column to spread

values → values to fill

In [ ]:
df_pivot = df_melted.pivot(
    index='name',
    columns='subject',
    values='marks'
)

df_pivot